## Setup
Generated by `scripts/build_notebooks.py`; do not edit. Installs `whopays` at `WHOPAYS_REF`,
reads secrets, and prints the environment. Needs a GPU runtime (Colab: Runtime → Change
runtime type → T4 GPU. Kaggle: Settings → Accelerator → GPU T4 x2, and Internet on).

In [ ]:
WHOPAYS_REF = "main"  # @param {type:"string"}
MOUNT_DRIVE = False  # @param {type:"boolean"}
# Offline only: exact path of a wheel from `build_notebooks.py --bundle`, e.g.
# /content/whopays-0.1.0-py3-none-any.whl (no quotes in the form).
# Empty = install from GitHub at WHOPAYS_REF.
WHOPAYS_WHEEL = ""  # @param {type:"string"}
REPO = "SVamseekar/whopays"

import os, subprocess, sys  # noqa: E401

# Colab string fields can preserve quotation marks pasted with a Python path literal.
WHOPAYS_WHEEL = WHOPAYS_WHEEL.strip()
if (len(WHOPAYS_WHEEL) >= 2 and WHOPAYS_WHEEL[0] in "\"'"
        and WHOPAYS_WHEEL[-1] == WHOPAYS_WHEEL[0]):
    WHOPAYS_WHEEL = WHOPAYS_WHEEL[1:-1]

# Kaggle first: Kaggle images are built on Colab's and also set COLAB_RELEASE_TAG.
PLATFORM = ("kaggle" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ
            else "colab" if "COLAB_RELEASE_TAG" in os.environ else "local")
_gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True)
if _gpu.returncode != 0:
    raise SystemExit("No CUDA GPU attached. Colab: Runtime -> Change runtime type -> T4 GPU. "
                     "Kaggle: Settings -> Accelerator -> GPU T4 x2.")
print("platform:", PLATFORM, "| GPU:", _gpu.stdout.strip().replace("\n", ", "))


def _secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        if PLATFORM == "colab":
            from google.colab import userdata
            return userdata.get(name)
        if PLATFORM == "kaggle":
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
    except Exception as exc:
        print(f"secret {name} not available ({type(exc).__name__})")
    return None


def _sh(*cmd, **kw):
    subprocess.run(cmd, check=True, **kw)


for _name in ("HF_TOKEN", "GH_TOKEN"):
    if (_value := _secret(_name)):
        os.environ[_name] = _value

if WHOPAYS_WHEEL:  # opt-in offline fallback: exact path to a wheel you built and uploaded
    if not (os.path.isfile(WHOPAYS_WHEEL) and os.path.basename(WHOPAYS_WHEEL).startswith("whopays-")
            and WHOPAYS_WHEEL.endswith(".whl")):
        raise SystemExit(f"WHOPAYS_WHEEL={WHOPAYS_WHEEL!r} is not a whopays-*.whl file")
    import hashlib, zipfile  # noqa: E401
    with zipfile.ZipFile(WHOPAYS_WHEEL) as _archive:
        _locks = [n for n in _archive.namelist()
                  if n.endswith(".data/data/share/whopays/requirements.lock")]
        if len(_locks) != 1:
            raise SystemExit("Wheel has no unique embedded requirements.lock; rebuild it")
        _lock = _archive.read(_locks[0]).decode("utf-8")
    with open(WHOPAYS_WHEEL, "rb") as _wheel:
        _wheel_code = "wheel:sha256:" + hashlib.file_digest(_wheel, "sha256").hexdigest()
    if "whopays" in sys.modules and os.environ.get("WHOPAYS_CODE_VERSION") != _wheel_code:
        raise SystemExit("Restart the runtime before loading a different WhoPays build")
    _keep = ("numpy==", "pandas==", "pyarrow==", "scipy==", "matplotlib==")
    with open("/tmp/whopays-constraints.txt", "w") as _constraints:
        _constraints.writelines(line for line in _lock.splitlines(keepends=True)
                                if not line.startswith(_keep))
    _sh(sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/whopays-constraints.txt",
        WHOPAYS_WHEEL + "[gpu]")
    # pip can skip a rebuilt wheel with the same package version. Reinstall only our code.
    _sh(sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
        WHOPAYS_WHEEL)
    os.environ["WHOPAYS_CODE_VERSION"] = _wheel_code
else:
    _src = "/content/whopays" if PLATFORM == "colab" else "/kaggle/working/whopays-src"
    if os.environ.get("GH_TOKEN"):  # private repo; keeps the token out of URLs and logs
        os.environ.update(GIT_CONFIG_COUNT="1",
                          GIT_CONFIG_KEY_0=f"url.https://x-access-token:{os.environ['GH_TOKEN']}"
                                           "@github.com/.insteadOf",
                          GIT_CONFIG_VALUE_0="https://github.com/")
    if not os.path.isdir(_src):
        _sh("git", "init", "-q", _src)
        _sh("git", "-C", _src, "remote", "add", "origin", f"https://github.com/{REPO}.git")
    _sh("git", "-C", _src, "fetch", "-q", "--depth", "1", "origin", WHOPAYS_REF)
    _sh("git", "-C", _src, "checkout", "-q", "--force", "FETCH_HEAD")
    _commit = subprocess.run(["git", "-C", _src, "rev-parse", "HEAD"], capture_output=True,
                             text=True, check=True).stdout.strip()
    if "whopays" in sys.modules and os.environ.get("WHOPAYS_CODE_VERSION") != _commit:
        raise SystemExit("Restart the runtime before loading a different WhoPays build")
    # Pin our stack, but keep the platform's own numpy/pandas/pyarrow/scipy (like torch):
    # replacing them breaks the preinstalled Colab/Kaggle packages that depend on them.
    _keep = ("numpy==", "pandas==", "pyarrow==", "scipy==", "matplotlib==")
    with open(f"{_src}/requirements.lock") as f, open("/tmp/whopays-constraints.txt", "w") as g:
        g.writelines(line for line in f if not line.startswith(_keep))
    _sh(sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/whopays-constraints.txt",
        f"{_src}[gpu]")
    os.environ["WHOPAYS_CODE_VERSION"] = _commit

if MOUNT_DRIVE and PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["WHOPAYS_RESULTS"] = "/content/drive/MyDrive/whopays-results"

import whopays.env  # noqa: E402

whopays.env.prepare_text_runtime()
whopays.env.print_env()

# 05 · Exactness gate and speed grid (FR-5)

1. **Gate:** speculative output must equal plain greedy, token for token, across 110 prompts
   (10 per language) under every cache policy, running past EOS so flushes occur. This is the
   T4 evidence compressed-cache block verification waits for.
2. **Speed grid:** first 30 MGSM items per language, draft off and on, batch size 1.

**Needs a K1 pass.** **Send back:** the gate verdict and the tables.

In [ ]:
import pandas as pd

import whopays
from whopays.config import load_jobs
from whopays.decoding.exactness import run_gate
from whopays.env import results_root
from whopays.gates import require_k1
from whopays.grids import K1_MODEL, speed_grid, speed_subset

require_k1(results_root())
gate, state = run_gate(K1_MODEL)
print(gate.to_string(index=False))
print("exactness gate:", state)
if state == "FAIL":
    raise SystemExit("Speculative output diverged from greedy: a harness bug. Do not run the grid.")

In [ ]:
JOBS = load_jobs(speed_grid() + speed_subset())
results = whopays.run(JOBS)
status = pd.DataFrame([r.__dict__ for r in results])
print(status.groupby("status").size())

In [ ]:
from whopays.io.results import load_runs

rows = load_runs(results_root(), JOBS)
summary = rows.groupby(["model", "weight", "kv", "spec", "lang"], as_index=False).agg(
    items=("item_id", "size"), wall_s=("wall_time_s", "sum"), output=("output_tokens", "sum"))
summary["tokens_per_s"] = summary.output / summary.wall_s
print(summary.round(2).to_string(index=False))